## Split GDPR file into separate article files

In [1]:
from pathlib import Path
import re

source = Path("/Users/davidj.silva/Desktop/Data Science/Thesis/Project/Dataset/Reference/Reference Law - GDPR.txt")
target_dir = Path("/Users/davidj.silva/Desktop/Data Science/Thesis/Project/Dataset/Reference")
target_dir.mkdir(exist_ok=True)

raw_text = source.read_text(encoding="utf-8", errors="ignore")
parts = re.split(r"(?=Article\s+\d+)", raw_text)

for part in parts:
    part = part.strip()
    if not part:
        continue
    match = re.match(r"Article\s+(\d+)\s*[-–]?\s*(.*)", part)
    if match:
        num = match.group(1)
        title = match.group(2).strip()
        text = re.sub(r"Article\s+\d+\s*[-–]?\s*.*", "", part, count=1).strip()
        out_file = target_dir / f"Article_{num}.txt"
        out_file.write_text(text, encoding="utf-8")
        print(f"Saved: {out_file.name}")


Saved: Article_16.txt
Saved: Article_8.txt
Saved: Article_16.txt
Saved: Article_16.txt
Saved: Article_2.txt
Saved: Article_25.txt
Saved: Article_26.txt
Saved: Article_47.txt
Saved: Article_263.txt
Saved: Article_263.txt
Saved: Article_263.txt
Saved: Article_263.txt
Saved: Article_267.txt
Saved: Article_267.txt
Saved: Article_263.txt
Saved: Article_11.txt
Saved: Article_179.txt
Saved: Article_338.txt
Saved: Article_17.txt
Saved: Article_290.txt
Saved: Article_5.txt
Saved: Article_28.txt
Saved: Article_1.txt
Saved: Article_2.txt
Saved: Article_98.txt
Saved: Article_3.txt
Saved: Article_4.txt
Saved: Article_27.txt
Saved: Article_51.txt
Saved: Article_1.txt
Saved: Article_5.txt
Saved: Article_89.txt
Saved: Article_89.txt
Saved: Article_6.txt
Saved: Article_23.txt
Saved: Article_9.txt
Saved: Article_10.txt
Saved: Article_7.txt
Saved: Article_8.txt
Saved: Article_6.txt
Saved: Article_9.txt
Saved: Article_89.txt
Saved: Article_10.txt
Saved: Article_6.txt
Saved: Article_11.txt
Saved: Article_1

## Environment setup & imports

In [2]:
# ===============================================================
# 1. ENVIRONMENT SETUP & IMPORTS
# ===============================================================
import os
import sys
import logging
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- FIX: add your project root to sys.path so "src" is importable ---
project_root = Path("/Users/davidj.silva/Desktop/Data Science/Thesis/Project")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"✅ Added project root to sys.path: {project_root}")

# --- import your modular code ---
from src.data_loader import load_datasets
from src.cleaning_utils import clean_text
from src.agents import SensitiveDataAgent, PolicyAuditAgent
from src.visualization import create_dashboard
from src.ml_model import train_baseline_model

✅ Added project root to sys.path: /Users/davidj.silva/Desktop/Data Science/Thesis/Project


## LOGGING CONFIGURATION

In [3]:
# ===============================================================
# 2. LOGGING CONFIGURATION
# ===============================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logging.info("✅ Environment ready and src modules loaded.")

2025-10-24 21:27:39,478 - INFO - ✅ Environment ready and src modules loaded.


## Load Consent Forms

In [4]:
# ===============================================================
# 3. LOAD DATASETS
# ===============================================================
CONSENT_PATH = Path("/Users/davidj.silva/Desktop/Data Science/Thesis/Project/Dataset/Consent Forms")
REFERENCE_PATH = Path("/Users/davidj.silva/Desktop/Data Science/Thesis/Project/Dataset/Reference")

df = load_datasets(CONSENT_PATH, REFERENCE_PATH)
df["clean_text"] = df["text"].apply(clean_text)
print("Total documents loaded:", len(df))
print("\nBy label:")
print(df["label"].value_counts())
logging.info(f"Loaded dataset with {len(df)} documents.")
df.head()

2025-10-24 21:27:39,778 - INFO - Loaded dataset with 219 documents.


Total documents loaded: 219

By label:
label
consent_form     114
reference_law    105
Name: count, dtype: int64


,file_name,text,label,clean_text
0,help_instagram_com_privacy.txt,Meta Privacy Policy - How Meta collects and us...,consent_form,meta privacy policy - how meta collects and us...
1,www_figma_com_privacy.txt,Privacy Policy\nSkip to main content\nProducts...,consent_form,privacy policy skip to main content products s...
2,www_webmd_com_privacy.txt,WebMD Privacy Policy\nSkip to main content\nHo...,consent_form,webmd privacy policy skip to main content home...
3,www_hsbc_com_privacy.txt,Privacy notices | Safeguard your data | HSBC H...,consent_form,privacy notices safeguard your data hsbc hol...
4,www_marriott_com_privacy.txt,Privacy Center\nSkip to Content\nOpen Menu\nMa...,consent_form,privacy center skip to content open menu marri...


## Basic Text Cleaning & Tokenization

In [5]:
# ===============================================================
# 4. RUN AGENTS
# ===============================================================
sda = SensitiveDataAgent()
paa = PolicyAuditAgent()

def process_form(row):
    text = row["clean_text"]
    return {
        "file_name": row["file_name"],
        "label": row["label"],
        "pii_found": sda.detect(text),
        "audit_score": paa.evaluate(text)
    }

results = [process_form(row) for _, row in df.iterrows()]
results_df = pd.DataFrame(results)

In [6]:
from src.gdpr_agent import GDPRComplianceAgent

# --- evaluate ONLY consent forms for detailed audit ---
df_consent_only = df[df["label"] == "consent_form"].copy()
df_consent_only = df_consent_only[df_consent_only["clean_text"].str.strip().str.len() > 0]
print("✅ Non-empty consent forms:", len(df_consent_only))

agent = GDPRComplianceAgent(gdpr_articles_df=None)  # later you can pass df_gdpr for citations

def audit_one(row):
    rep = agent.evaluate_policy(row["clean_text"])
    return {
        "file_name": row["file_name"],
        "detailed_report": rep,
        "overall_score": agent.overall_score(rep)
    }

audited = [audit_one(r) for _, r in df_consent_only.iterrows()]
audit_df = pd.DataFrame([{k:v for k,v in d.items() if k != "detailed_report"} for d in audited])

print("Top 10 files by overall GDPR compliance:")
display(audit_df.sort_values("overall_score", ascending=False).head(10))


✅ Non-empty consent forms: 112
Top 10 files by overall GDPR compliance:


,file_name,overall_score
13,www_netflix_com_privacy.txt,0.875000
19,www_shopify_com_privacy.txt,0.875000
75,aws_amazon_com_privacy.txt,0.875000
8,ec_europa_eu_privacy.txt,0.875000
88,www_atlassian_com_privacy.txt,0.833333
101,www_netlify_com_privacy.txt,0.833333
68,www_heroku_com_privacy.txt,0.833333
67,www_salesforce_com_privacy.txt,0.833333
80,www_trello_com_privacy.txt,0.833333
107,zoom_us_privacy.txt,0.791667


## COMPUTE SCORES & DISPLAY DASHBOARD

In [7]:
# ===============================================================
# 5. COMPUTE SCORES & DISPLAY DASHBOARD
# ===============================================================
from src.visualization import compute_score
results_df["score"] = results_df["audit_score"].apply(compute_score)

# Only keep consent forms in the dashboard
results_df = results_df[df["label"] == "consent_form"]

fig = create_dashboard(results_df, title="Consent Form Audit Dashboard — PII & Compliance Overview")
fig.show()


## ML Training

In [8]:
# ===============================================================
# 6. TRAIN BASELINE ML MODEL
# ===============================================================
from src.ml_model import train_baseline_model
vectorizer, clf = train_baseline_model(df)

import joblib
joblib.dump(vectorizer, project_root / "models" / "tfidf_vectorizer.pkl")
joblib.dump(clf, project_root / "models" / "compliance_classifier.pkl")

print("✅ Baseline ML model trained and saved successfully.")


               precision    recall  f1-score   support

 consent_form       1.00      0.95      0.97        20
reference_law       0.96      1.00      0.98        24

     accuracy                           0.98        44
    macro avg       0.98      0.97      0.98        44
 weighted avg       0.98      0.98      0.98        44

✅ Model saved successfully at: /Users/davidj.silva/Desktop/Data Science/Thesis/Project/models/compliance_model.pkl
✅ Baseline ML model trained and saved successfully.


## SAVE RESULTS

In [9]:
# ===============================================================
# 7. SAVE RESULTS
# ===============================================================
output_path = Path("/Users/davidj.silva/Desktop/Data Science/Thesis/Project/Dataset/Results/consent_audit_results.csv")
results_df.to_csv(output_path, index=False)
logging.info(f"💾 Results saved to {output_path}")

2025-10-24 21:27:45,125 - INFO - 💾 Results saved to /Users/davidj.silva/Desktop/Data Science/Thesis/Project/Dataset/Results/consent_audit_results.csv
